<div style="margin: 0 auto 10px; height: 70px; border: 2px solid gray; border-radius: 6px;">
  <div style="float: left; margin: 5px 10px 5px 10px; "><img src="img/bfh.jpg" /></div>
  <div style="float: right; margin: 20px 30px 0; font-size: 15pt; font-weight: bold; color: #98b7d2;"><a href="https://moodle.bfh.ch/course/view.php?id=39255" style="color: #98b7d2;">BTE5476 - Project-Oriented Digital Signal Processing </a></div>
</div>
<div style="clear: both; font-size: 30pt; font-weight: bold; color: #64788b; margin-left: 30px;">
    Phase-Locked Loops (with applications)
</div>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.signal as sp
import IPython
from scipy.io import wavfile

In [ ]:
plt.rcParams["figure.figsize"] = (9,2.5)

The following functions were originally defined in the notebooks on biquad filters.

In [ ]:
def LPF(fc, sf, Q=(1/np.sqrt(2))):
    """Biquad lowpass filter"""
    w = 2 * np.pi * fc / sf
    alpha = np.sin(w) / (2 * Q)
    c = np.cos(w)
    a = np.array([1 + alpha, -2 * c, 1 - alpha])
    b = np.array([(1 - c) / 2, 1 - c, (1 - c) / 2])
    return b / a[0], a / a[0]
    
def BPF(fc, bw, sf):
    """Biquad bandpass filter"""
    w = 2 * np.pi * fc / sf
    alpha = np.tan(np. pi * bw / sf)
    c = np.cos(w)
    b = np.array([alpha, 0, -alpha])
    a = np.array([1 + alpha, -2 * c, 1 - alpha])
    return b / a[0], a / a[0]

def NTCF(fc, bw, sf):
    """Biquad notch filter"""
    w = 2 * np.pi * fc / sf
    alpha = np.tan(np. pi * bw / sf)
    c = np.cos(w)
    b = np.array([1, -2 * c, 1])
    a = np.array([1 + alpha, -2 * c, 1 - alpha])
    return b / a[0], a / a[0]

In [ ]:
def analog_response(b, a, sf, dB=-100, half=True, points=2001, axis=None, color='C0', label=None):
    """plot the magnitude response of a digital filter on a log-log scale"""
    EPS = 1e-20
    w = np.linspace(0, np.pi, points)
    A, B = np.zeros(points, dtype='complex'), np.zeros(points, dtype='complex')
    for n, bn in enumerate(b):
        B += bn * np.exp(-1j * n * w)
    for n, an in enumerate(a):
        A += an * np.exp(-1j * n * w)
    M = 20 * np.log10(np.abs(np.where(B == 0, EPS, B) / (np.where(A == 0, EPS, A))))
    
    if axis is None:
        _, axis = plt.subplots()
    axis.plot(np.linspace(0, sf / 2, points), M, color, lw=2, label=label)
    axis.set_xscale('log')
    axis.set_ylabel('dB', color='C0')
    axis.grid(visible=True, which='both')    
    axis.axis('tight')
    axis.set_ylim([max(dB, min(M)), None])  

In [ ]:
def plot_mag_spec(data, Fs=2*np.pi, pts=None, maxf=None, minf=None, one_sided=True):
    pts = len(data) if pts is None else pts
    w = Fs * (np.arange(0, pts) / pts - 0.5)
    X = np.abs(np.fft.fftshift(np.fft.fft(data, pts)))
    maxf = Fs / 2 if maxf is None else maxf
    minf = -Fs / 2 if minf is None else minf
    if one_sided and minf < 0:
        minf = 0
    start = max(0, int(pts * (Fs / 2 + minf) / Fs))
    stop = min(pts, int(pts * (Fs / 2 + maxf) / Fs))
    plt.plot(w[start:stop], X[start:stop]);

In [ ]:
Fs = 32000  # default sampling rate for the examples in the notebook

# Phase-Locked Loops

A phase-locked-loop (PLL) is an adaptive device that can precisely track a periodic input signal; it does so by adjusting the phase and the frequency of an internal oscillator so that they match the (potentially time-varying) phase and frequency of the input. The internal structure of a PLL, shown in the following diagram, is comprised of:

 * an NCO (Numerically Controlled Oscillator) that generates a local periodic input signal $\hat{x}[n]$ as a function of a feedback control signal $c[n]$
 * a phase offset estimator that compares the input signal $x[n]$ to the NCO's output $\hat{x}[n]$
 * a _loop filter_ $L(z)$, that derives the NCO control signal $c[n]$ from the phase offset estimations 

<div style="margin: 0 auto;"><img src="img/PLL.png" width="600"></div>



## The NCO

<div style="float:right; margin: 0 10px;"><img src="img/nco.png" width="400"></div>

A Numerically Controlled Oscillator (NCO) is a stateful (i.e. _with memory_) discrete-time device with a user-definable default frequency value $\omega_c$. 

At every time $n$, the NCO's output is $y[n] = p(\varphi[n])$, where  $\varphi[n]$ is the NCO's internal phase and where $p(\cdot)$ is a $2\pi$-periodic function. When used in a PLL, $p(\cdot)$ is usually chosen to have the same shape as the input signal; if the input is a sinusoidal signal, for instance, then $p(\cdot)$ is also a sinusoidal function.

The NCO's internal phase $\varphi[n]$ is updated at each step as $$\varphi[n] = \varphi[n-1] + \omega_c + c[n];$$ this corresponds to applying an ideal integrator to the sequence $\omega_c + c[n]$. Since the total phase of an oscillation is the integral of its frequency over time, we can consider $\omega_c + c[n]$ as the NCO's instantaneous _frequency_.

### A sinusoidal NCO

If we choose $p(\cdot) = \sin(\cdot)$, and assuming zero initial conditions, the NCO's output at time $n$ is 

$$
    y[n] = \sin\left(\varphi[n]\right) = \sin\left(\sum_{k=0}^{n} (\omega_c + c[k])\right) = \sin\left(\omega_c\, n + \sum_{k=0}^{n}c[k]\right)
$$

In particular:

 * if $c[n] = 0$, the NCO will produce a sinusoid at the nominal frequency $\omega_c$
 * if $c[n] = \theta_0 \delta[n-n_0]$, the NCO's output will undergo a phase shift by $\theta_0$ radians at $n = n_0$
 * if $c[n] = \omega_0$, the NCO will produce a sinusoid at frequency $\omega_c + \omega_0$

## Phase offset detection

The phase offset between the input signal and the NCO's output can be computed in many different ways depending on the type of waveform that the PLL is designed to track. The good news is that, thanks to the PLL's feedback structure, everything will work just fine even if the phase estimation is relatively coarse.

With sinusoidal signals, for instance, offset estimation is typically implemented simply by multiplying the NCO's output with the input signal. To see why this works, let's express the input as $x[n] = \cos\alpha[n]$ and the NCO's output as $y[n] = \sin\varphi[n]$ so that the phase offset is $\theta[n] = \varphi[n] - \alpha[n] + \pi/2$; if we multiply the two signals we have

$$
    2x[n]y[n] = 2\sin\varphi[n]\cos\alpha[n] = \sin(\varphi[n]-\alpha[n]) + \sin(\varphi[n]+\alpha[n]) = \sin\theta[n] + \sin(\varphi[n]+\alpha[n])
$$

When the NCO is almost in lock with the input, the phase offset will be small and therefore $\sin\theta[n] \approx \theta[n]$; and so 

$$
    2x[n]y[n] \approx \theta[n] + \sin(\varphi[n]+\alpha[n]);
$$

the last term a high-frequency oscillation that will be attenuated by the lowpass characteristic of the loop filter. Note that, when the PLL is in lock, the NCO's output is always _in quadrature_ with the input, that is, if the input is a cosine, the NCO will produce a sine. To provide a clean copy of the input, generally NCOs also output a secondary signal with a $\pi/2$ phase shift.

## The Loop filter

<div style="float:right; margin: 10px 0;"><img src="img/loopfilter.png" width="400"></div>

The Loop Filter is the most delicate component of a PLL and designing a good loop filter for a given use case is quite an art. In its most general form, a loop filter is equivalent to a [PID controller](https://en.wikipedia.org/wiki/PID_controller) although in most cases only the Proportional and Integral components are present, while the Derivative part is omitted. Such a loop filter produces an NCO control signal of the form

$$
    c[n] = G_p\, \theta[n] + G_f\,\omega[n]
$$

where 

$$
    \omega[n] = \omega[n-1] + \theta[n] \\ 
$$

The control signal will thus include:
 * a proportional term, depending on the instantaneous phase offset via a _phase gain_ factor $G_p$; in a PLL the proportional term can compensate a phase offset between input and output but it won't be sufficient to track a frequency offset;
 * an integral term $\omega[n]$, computed from the instantaneous phase offset via an ideal integrator, and scaled by $G_f$; from the relation between phase and frequency, we can interpret $G_f$ as a _frequency_ gain factor and, in a PLL, this term is necessary to track frequency offsets.

We won't go into the theoretical details on how to determine the optimal values of $G_p$ and $G_f$ for a given application but, in a nutshell,

  * $G_p$ and $G_f$ must both be less than one in magnitude to ensure stability (both are usually very small) 
  * $G_p$ determines the "sensitivity" of the PLL to instantaneous phase offsets
  * $G_f$ affects the _bandwidth_ of the PLL, that is, the interval of frequencies around the nominal value $\omega_c$ that the PLL can successfully track  

## Exercise: a PLL for sinusoidal signals

Complete the following code block so that it implements a PLL for sinusoidal inputs. The ``track`` function should return the PLL's output as well as the control signal fed to the NCO.

In [ ]:
class PLL:
    def __init__(self, wc: float, gp: float, gf: float):
        # nominal frequency in radians
        self.wc = wc          
        # phase and frequency gains
        self.gp, self.gf = gp, gf
        # loop filter delays
        self.theta = 0
        self.phi = 0
        
    def track(self, x: np.ndarray) -> (np.ndarray, np.ndarray):
        # returns local replica of input and NCO control signal
        y = np.zeros(len(x))  
        c = np.zeros(len(x)) 
        
        self.theta = self.phi = 0 # reset
        for n in range(0, len(x)):
            # ... your code here
            pass
        return y, c

This utility function allows us to easily test the behavior of the PLL as in the examples that follow

In [ ]:
def test_PLL(wc: float, x: np.ndarray, phase_gain: float, freq_gain: float, 
             show_range=None, # if positive, show first N output values, if negative show last N 
             show_control=False, show_error=False,
             pll_type=PLL  # this will be useful later, ignore for now
            ):
    if show_range is None:
        n = np.arange(0, len(x))
    elif show_range > 0:
        n = np.arange(0, show_range)
    else:
        n = np.arange(len(x) + show_range, len(x))
    
    y, c = pll_type(wc, phase_gain, freq_gain).track(x)

    plt.plot(n, x[n], 'C0', label='input')    
    plt.plot(n, y[n], 'C1', label='output')
    if show_error:
        plt.plot(n, 0.5 * (x-y)[n], 'C3', label='error (x0.5)')
    if show_control:
        plt.plot(n, 10 * c[n], 'C2:', label='control (x10)')
    plt.legend(loc="lower right")

Let's also define a couple of sinusoidal test signals via this helper function

In [ ]:
def test_signal(wc, phase_shift=0, freq_dev_perc=0, samples=2000, gain=1, SNR=1e6): 
    x = np.sin((1 + freq_dev_perc / 100) * wc * np.arange(0, samples) + phase_shift) * gain
    # power of sinusoid is gain^2/2; power of Gaussian noise scaled by a is a^2
    a = np.sqrt(0.5 * gain * gain * np.power(10, -SNR / 10))
    x += a * np.random.randn(len(x))
    return wc, x

In [ ]:
wc = 1.05 # ~ 2 * np.pi / 6
# phase shift only
test_p = test_signal(wc, np.pi)
# phase shift and 5% frequency deviation
test_pf = test_signal(wc, np.pi, 5)

### Open loop

With both gains set to zero, the PLL's output is simply a sinusoid at PLL's nominal frequency. Here the input has both a phase offset and a frequency deviation and it's clear that input and output are moving independently, with no tracking taking place 

In [ ]:
test_PLL(*test_pf, phase_gain=0, freq_gain=0, show_range=200)

### Phase offset compensation

Here the input has a 90-degree initial phase offset and no frequency deviation; only a phase gain is required for the PLL to lock with the input over time

In [ ]:
test_PLL(*test_p, phase_gain=0.02, freq_gain=0, show_range=200, show_error=True, show_control=True)

We can see the rate of convergence more clearly over a longer time span (please be aware that the control signal is shown magnified 10 times to be visible)

In [ ]:
test_PLL(*test_p, phase_gain=0.02, freq_gain=0, show_error=True, show_control=True)

Note how the control signal keeps oscillating even though the error goes to zero; this is due to the high-frequency term in the multiplication-based phase estimation method but it does not prevent the PLL from successfully locking since the average value of the control signal is indeed zero.

### Frequency tracking

Here the input has an initial phase offset and a 5% frequency deviation; with the frequency gain set to zero the PLL cannot get a lock no matter how long we run it

In [ ]:
test_PLL(*test_pf, phase_gain=0.02, freq_gain=0, show_range=-300, show_error=True, show_control=True)

To achieve a lock when the input's frequency deviates from its nominal value, we must set the frequency gain $G_f$ to a positive value:

In [ ]:
test_PLL(*test_pf, phase_gain=0.02, freq_gain=0.02, show_range=-300, show_error=True, show_control=True)

Note how in this case the control signal converges (in average) to the theoretical value of the frequency offset, i.e. 0.05 (don't forget the control signal is shown magnified 10 times)

In [ ]:
test_PLL(*test_pf, phase_gain=0.02, freq_gain=0.01, show_error=True, show_control=True)

The PLL's lock range is determined by the value of $G_f$; in this example, the frequency deviation is too large and the PLL cannot achieve a lock

In [ ]:
test_large_dev = test_signal(wc, np.pi, 35)
test_PLL(*test_large_dev, phase_gain=0.02, freq_gain=0.01, show_range=-300, show_error=True)

We can widen the lock frequency range by increasing $G_f$:

In [ ]:
test_PLL(*test_large_dev, phase_gain=0.02, freq_gain=0.04, show_range=-300, show_error=True, show_control=True)

But a large value for $G_f$ is problematic if the PLL is designed with a small nominal frequency: the increased gain makes the loop filter too sensitive to the instantaneous phase estimation error and, as a consequence, with low-frequency sinusoids the NCO will be slowed down when the input is small (near zero crossings) and accelerate when the input peaks, resulting in a severely distorted output even with no frequency deviation in the input:

In [ ]:
test_slow = test_signal(wc / 10, np.pi, 0)
test_PLL(*test_slow, phase_gain=0.02, freq_gain=0.04, show_range=-500)

Reducing the frequency gain fixes the problem: bottom line, choosing the right gain values can be rather tricky and there is always a tradeoff somewhere!

In [ ]:
test_PLL(*test_slow, phase_gain=0.02, freq_gain=0.004, show_range=-500)

## Robustness

PLLs are remarkably robust to variations in input amplitude and to additive noise.

### Input amplitude

Sinusoidal PLLs can achieve lock across a good range of input amplitudes: 

In [ ]:
test_PLL(*test_signal(wc, np.pi, 5, gain=2), phase_gain=0.02, freq_gain=0.01, show_range=-200)

In [ ]:
test_PLL(*test_signal(wc, np.pi, 5, gain=0.4), phase_gain=0.02, freq_gain=0.01, show_range=-200)

### Performance in noise

Sinusoidal PLLs have excellent noise rejection and can still easily track an input signal even at a SNR of zero, i.e. when the power of the noise is equal to that of the signal.

In [ ]:
test_PLL(*test_signal(wc, np.pi, 5, SNR=10), phase_gain=0.02, freq_gain=0.01, show_range=-200)

In [ ]:
test_PLL(*test_signal(wc, np.pi, 5, SNR=0), phase_gain=0.02, freq_gain=0.01, show_range=-200)

# Remote Synchronization

Consider the problem of sending a synchronization event to a remote device; for example, imagine we need to "wake up" a data receiver to signal that a data payload is incoming, or that we need to synchronize the internal clock of the remote device to a specific time reference.

To do this, we need to design a signal with an identifiable feature that is well localized in time, and we need to make sure that this feature remains clearly identifiable after the signal is transmitted over the communication channel to the remote device.

## A simple transmission channel

Let's write a simple class that simulates a bandlimited channel with additive Gaussian noise:

In [ ]:
class BandpassChannel:
    def __init__(self, bw, SNR=1e6, fs=Fs):
        self.fs = fs
        self.center_freq = np.mean(bw)
        self.wc = 2 * np.pi * self.center_freq / fs
        self.ch_filt = sp.ellip(4, 3, 40, bw, 'bandpass', fs=fs)
        self.delay = int(sp.group_delay(self.ch_filt, w=[self.wc])[1][0] + .5)
        self.nl = np.power(10, -SNR / 20)

    def show(self):    
        analog_response(*self.ch_filt, self.fs, half=True)

    def transmit(self, x):
        y = sp.lfilter(*self.ch_filt, np.r_[ x, np.zeros(self.delay) ])[self.delay:] 
        return y + self.nl * np.random.randn(len(y))

    def plot_in_out(self, x):
        plt.subplot(121)
        plt.plot(x)
        plt.ylim([-1.2, 1.2])
        plt.title('transmitted')
        plt.subplot(122)
        plt.plot(self.transmit(x), 'C1')
        plt.ylim([-1.2, 1.2])
        plt.title('received')

In [ ]:
test_channel = BandpassChannel([1500, 2500], 25)
test_channel.show()

## Simple synchronization pulses

The most elementary type of time-localized feature is a sharp amplitude transition: imagine flipping a switch on or off.

Since every communication channel introduces at least some additive noise, in the absence of other impairments we just need to make sure that the sync pulse rises well above the noise floor; this way, the receiver can determine the timing of the pulse via simple thresholding. This may work if the channel has a very wide bandwidth...

In [ ]:
sync_len = 8000
sync_time = 4000

delta = np.zeros(sync_len)
delta[sync_time] = 1
BandpassChannel([100, Fs / 2 - 100], SNR=25).plot_in_out(delta)

... but it is not going to work the moment we send a synchronization pulse over a bandlimited channel:

In [ ]:
test_channel.plot_in_out(delta)

We could try to make the pulse wider, or ever use a step signal, but things are not going to improve much: 

In [ ]:
pulse = np.zeros(sync_len)
pulse_len = 100
pulse[sync_time:sync_time + pulse_len] = 1
test_channel.plot_in_out(pulse)

In [ ]:
step = np.zeros(sync_len)
step[sync_time:] = 1
test_channel.plot_in_out(step)

The problem, of course, is that the energy associated to time-localized sync pulse is spread out over all frequencies; since only a small portion of that energy "survives" the channel, at the receiving end the amplitude of the transition will be much smaller and will fall under the noise floor. We can see this clearly if we remove the noise:

In [ ]:
BandpassChannel(bw=[1500, 2500], SNR=1e6).plot_in_out(delta)

## Modulated pulses

A better approach is to match the synchronization signal to the channel's bandwidth by using modulated pulses; this way, most of the energy associated to the transition will not be attenuated during transmission:

In [ ]:
mod_pulse = np.zeros(sync_len)
mod_pulse[sync_time:sync_time+pulse_len] = np.cos(test_channel.wc * np.arange(0, pulse_len))
test_channel.plot_in_out(mod_pulse)

The main problem with such sync pulses, however, is that they produce a signal with very high _peak-to-average_ energy ratio; this may be problematic on power-limited channels such as a telephone line or on a channel with nonlinear characteristics. Additionally, short energy bursts do not play well with the automatic gain control mechanism implemented at the front-end of the receiver; ideally, we would like a sync signal with a constant amplitude envelope.

## Phase reversal  

A much more robust solution is to encode the timing reference as a 180-degree _phase reversal_ in a relatively long sinusoidal _pilot tone_ at a frequency well within the channel's passband: when detecting such a tone, the receiver can use it to calibrate its input gain before the phase reversal occurs.

In [ ]:
def sync_signal(len_sec, tone_freq, rev_time=0.5, sf=Fs):
    N = int(len_sec * sf)
    C = int(N * rev_time)
    wp = 2 * np.pi * tone_freq / sf 
    tail = np.cos(wp * np.arange(N))
    return np.r_[np.flipud(tail)[C:], -tail[1:N-C]]

In [ ]:
plt.plot(sync_signal(0.01, 600));

A simple method to detect a phase reversal is to first use a bandpass filter to isolate the tone, and then pass the result to a notch filter centered on the tone frequency. After a brief lead time, most of the tone energy will be "killed" by the notch but, when the phase reversal occurs, the signal's energy will momentarily spread out in frequency and will no longer be attenuated by the notch. The resulting energy burst can be easily detected even in the presence of noise:

In [ ]:
def notch_detector(channel, notch_freq=None, len_sec=0.1, sf=Fs):
    tone_freq = channel.center_freq
    if notch_freq is None:
        notch_freq = tone_freq
    rx_tone = channel.transmit(sync_signal(len_sec, tone_freq, sf=sf))
    notch_output = sp.lfilter(*NTCF(notch_freq, 100, sf), rx_tone)
    skip = int(len(notch_output) * .1)
    plt.plot(rx_tone[skip:], label="receiver's input");
    plt.plot(notch_output[skip:], 'C1', label="notch filter output");
    plt.legend();

In [ ]:
notch_detector(test_channel)

The problem with this simple approach is that the location of the notch must match the tone frequency very closely; if there is a mismatch with the frequency of the pilot tone, it will be hard or impossible to detect the change in energy associated to the phase reversal:

In [ ]:
notch_detector(test_channel, notch_freq=test_channel.center_freq * 0.9)

# Remote synchronization using PLLs

PLL's are very good at detecting sinusoids buried in noise and are thus ideal candidates to improve our synchronization mechanism based on phase reversals in a pilot tone.

## Tone detection

Assume PLL is tracking a sinusoidal input and let's consider the product between the PLL's input and its output; we have two possibilities:

 * if the PLL is in lock, input and output are in phase and so the product will always be positive
 * if the PLL is not in lock, the sign of the product will keep changing until lock is achieved

If we compute a running average of the input-output product, the result will be positive when the PLL is in lock and close to zero when the PLL is out of lock; this can be used to determine the PLL's status.

The following function sets up a PLL with a small frequency gain to ensure a narrow lock range around its nominal frequency and uses a simple lowpass to compute the local average of the input-output product; by thresholding this average we can obtain a lock indicator signal.

To avoid having to adjust the threshold as a function of the unknown amplitude of the input signal, we multiply the PLL's output (which is unit amplitude by design) with the _sign_ of the input sinusoid; this makes the system robust to variations in input level with no need to use any form of adaptive gain control.

In [ ]:
class ToneDetector(PLL):
    def __init__(self, tone_freq, phase_gain=0.06, freq_gain=0.001, sf=Fs):
        super(ToneDetector, self).__init__(2 * np.pi * tone_freq / sf, phase_gain, freq_gain)
        # we average the input-output product over K periods of the input sinusoid
        K = 5
        self.ma_len = int(sf / tone_freq * 5)

    def reveal(self, x):
        y, c = self.track(x)
        r = sp.lfilter(np.ones(self.ma_len), [1], y * np.sign(x)) / self.ma_len
        # compensate for the filter's delay
        return np.r_[ r[self.ma_len//2:], np.zeros(self.ma_len // 2) ]

    def detect_lock(self, x, threshold=0.25, plot=False):
        r = self.reveal(x)
        d = r > threshold
        if plot:
            plt.plot(x, 'C0');
            plt.plot(r, 'C1');
            plt.plot(d, 'C3');
        return d

Let's test the tone detector on a clean tone, a noisy tone, and a clean tone at the wrong frequency. Note how the PLL has little trouble achieving lock even when the signal to noise ratio is very low (zero dB in this example); this robustness to noise (which is inversely related to the lock range) makes PLLs ideal devices for signal detection. At the same time, the small frequency gain ensures that there is no lock with the last mismatched tone.

In [ ]:
tone_freq = 2000

N = int(Fs * 0.01)
w = 2 * np.pi * tone_freq / Fs
x = np.r_[np.zeros(N), 
    np.sin(w * np.arange(N)), np.zeros(N), 
    np.sin(w * np.arange(N)) + 0.7 * np.random.randn(N), np.zeros(N), 
    np.sin(0.7 * w * np.arange(N)), np.zeros(N)
]

detector = ToneDetector(tone_freq)
detector.detect_lock(x, plot=True);

## Phase reversal detection

If a PLL in lock state encounters a phase reversal in the input signal, the PLL's output will momentarily be inverted with respect to the input (i.e. input and output will have opposite signs) so that the local average of the input-output product becomes negative until the PLL recovers lock. The duration of this out-of-lock interval  depends on the phase gain $G_p$, which needs to be relatively small to ensure detecion of a phase reversal. 

In [ ]:
class SyncDetector(ToneDetector):
    def __init__(self, tone_freq, phase_gain=0.06, freq_gain=0.001, sf=Fs):
        super(SyncDetector, self).__init__(tone_freq, phase_gain, freq_gain, sf)

    def detect_sync(self, x, plot=False):
        r = self.reveal(x)
        s = np.sign(r)
        zc = (np.r_[ [0], s[:-1] ] > 0) * np.r_[ s[1:], [0] ]
        zc = zc < 0
        if plot:
            plt.plot(x, 'C0');
            plt.plot(r, 'C1');
            plt.plot(zc, 'C3');
        return zc

In [ ]:
tone_freq = 2000
sync = sync_signal(0.05, tone_freq)

detector = SyncDetector(tone_freq)
detector.detect_sync(sync, plot=True);

Reasonable frequency variations are successfully handled even with a small frequency gain:

In [ ]:
sync_fdev = sync_signal(0.05, tone_freq * 0.9)
detector.detect_sync(sync_fdev, plot=True);

A small value for the frequency gain $G_f$ results in a narrow lock range that automatically provides noise rejection, thereby eliminating the need for an initial bandpass filter around the tone frequency. In the following examples, the phase reversal is correctly detected even at a very low SNR:

In [ ]:
test_channel = BandpassChannel([1500, 2500], SNR=10)

detector.detect_sync(test_channel.transmit(sync), plot=True);

In [ ]:
detector.detect_sync(test_channel.transmit(sync_fdev), plot=True);

# Exercise: FM radio

## Radio modulation techniques

In wireless communications the goal is to encode the information of a baseband signal (such as music or voice) onto a high-frequency sinusoidal carrier; the result is a passband signal that matches the physical properties of the frequency band used for radio transmission. This spectral shift of the original baseband signal is called _modulation_.

The simplest type of radio transmission is Amplitude Modulation (AM), where the baseband signal is encoded in the carrier's amplitude:

$$
    s[n] = x[n] \, \cos(\omega_c \, n);
$$

while simple to implement, AM is however very sensitive to additive noise. 

<div style="float:right; margin: 10px;"><img src="img/amfm.png" width="400"></div>


Frequency Modulation (FM) was developed in the 1930s as a noise-resilient alternative (_"FM, no static at all"_); in FM the baseband information is encoded via small variations in the carrier's frequency:

$$
    s[n] = \cos\left(\sum_{k=0}^{n} (\omega_c + \gamma\, x[k])\right)
$$

where $\gamma$ is called the _modulation index_. FM is a nonlinear operation and therefore its theoretical analysis is difficult; it's easy to see, however, that an FM signal has a _constant envelope_ (that is, the amplitude is independent of $x[n]$) which makes it easier to reject additive noise. From the above expression, it's apparent that an FM modulator can be easily implemented digitally using an NCO; originally, in analog radio devices, a Voltage Controlled Oscillator (VCO) was used instead, to produce the analog FM signal 

$$
    s(t) = \cos\left(\int_{\tau=0}^{t} 2\pi f_c + \gamma\, x(\tau)\,\, d\tau\right).
$$

## Demodulation of FM signals 

Moving to discrete time, FM modulation can be easily implemented using a NCO. Even more interestingly, a digital FM signal can be easily demodulated by a single PLL with a narrow lock range around the FM carrier frequency.

The following audio file, sampled at 96 kHz, contains an FM-modulated music snippet; if you try to play the file directly you won't hear anything because the modulation process has shifted all the audio content outside of hearing range.

In [ ]:
fm_sf, fm_signal = wavfile.read("data/fm.wav")
print(f'signal sampled at {fm_sf} Hz')

In [ ]:
IPython.display.Audio(fm_signal, rate=fm_sf)

Can you design a PLL that recovers the original audio content?

# Exercise: a PLL for square waves

In the following exercise you are asked to develop a PLL that can operate on square waves instead of sinusoids. Here are some pointers to get you started:

 * the structure of the PLL will remain the same
 * you should modify the NCO so that it produces a square wave instead of a sinusoid; this only requires to change the function $p(\cdot)$ applied to the NCO's phase; for your convenience, below is an implementation that works like a standard trigonometric funcion and returns a balanced, symmetric, $2\pi$-periodic square wave
 * the critical part will be the phase offset estimator; because of the discontinuous and piecewise-constant shape of a square wave, you won't be able to use a simple multiplication-based estimator. You will need to look at the rising and falling edges of input and output to determine their relative alignment
 * the loop filter will remain the same, but you will need to manually determine new appropriate values for $G_p$ and $G_f$.

In [ ]:
def sqw(w):
    return  2 * (np.sin(w) > 0) - 1

def sqw_test_signal(wc, phase_shift=0, freq_dev_perc=0, samples=2000, gain=1.0, SNR=1e6): 
    x = gain * sqw((1 + freq_dev_perc / 100) * wc * np.arange(samples) + phase_shift)
    # power of square wave is gain^2; power of Gaussian noise is a^2 
    a = np.sqrt(gain * gain * np.power(10, -SNR / 10))
    x += a * np.random.randn(len(x))
    return wc, x    

In [ ]:
plt.stem(sqw_test_signal(2 * np.pi / 12, samples=50)[1]);

In [ ]:
class SQWPLL:
    def __init__(self, wc: float, gp: float, gf: float, sf=None):
        self.wc = wc
        self.gp, self.gf = gp, gf
        self.freq_offset = self.nco_phase = 0

    def track(self, x):
        y = np.zeros(len(x))  # NCO output
        c = np.zeros(len(x))  # control signal
        
        # your code here...
        
        return y, c

Once you complete the PLL code, you can run the following tests:

In [ ]:
# Phase offset compensation
wc = 0.25

test_PLL(*sqw_test_signal(wc, phase_shift=np.pi), pll_type=SQWPLL, phase_gain=0.1, freq_gain=0, show_range=400, show_error=False, show_control=True)

In [ ]:
test_PLL(*sqw_test_signal(wc, phase_shift=np.pi), pll_type=SQWPLL, phase_gain=0.05, freq_gain=0, show_error=True, show_control=True)

In [ ]:
# frequency compensation
test_PLL(*sqw_test_signal(wc, phase_shift=np.pi/4, freq_dev_perc=5), pll_type=SQWPLL, phase_gain=0.1, freq_gain=0.0005, show_range=300, show_control=True)

In [ ]:
test_PLL(*sqw_test_signal(wc, phase_shift=np.pi/4, freq_dev_perc=5), pll_type=SQWPLL, phase_gain=0.1, freq_gain=0.0005, show_error=True, show_control=True)

In [ ]:
# performance in noise
test_PLL(*sqw_test_signal(wc, phase_shift=np.pi, freq_dev_perc=5, SNR=10), pll_type=SQWPLL, phase_gain=0.1, freq_gain=0.0005, show_range=-300, show_control=True)